# Notebook 00: SAP_FIN_DB full snapshot to Bronze

Reads the complete 13-table source projection from the deployed Fabric SQL Database using the pipeline execution identity and writes one exact dated Bronze CSV per table.


In [ ]:
sql_server = ""
sql_database = ""
processing_date = ""


In [ ]:
from datetime import datetime
from pathlib import PurePosixPath
import uuid

TABLE_CONFIG = [{'name': 'ACDOCA', 'sinkFolder': 'bronze/sap_finance/acdoca', 'filePrefix': 'acdoca', 'columnList': '[RCLNT],[RLDNR],[RBUKRS],[GJAHR],[POPER],[BELNR],[DOCLN],[RACCT],[RCNTR],[PRCTR],[KOKRS],[LIFNR],[KUNNR],[BUDAT],[BLDAT],[DRCRK],[BLART],[AWTYP],[SGTXT],[RHCUR],[RKCUR],[RWCUR],[HSL],[KSL],[TSL],[WSL],[MSL]', 'sourceObject': '[SAPABAP1].[ACDOCA]'}, {'name': 'BKPF', 'sinkFolder': 'bronze/sap_finance/bkpf', 'filePrefix': 'bkpf', 'columnList': '[MANDT],[BUKRS],[GJAHR],[BELNR],[MONAT],[BUDAT],[BLDAT],[BLART],[STBLG],[BKTXT]', 'sourceObject': '[SAPABAP1].[BKPF]'}, {'name': 'T001', 'sinkFolder': 'bronze/sap_finance/t001', 'filePrefix': 't001', 'columnList': '[MANDT],[BUKRS],[BUTXT],[ORT01],[LAND1],[WAERS],[KTOPL]', 'sourceObject': '[SAPABAP1].[T001]'}, {'name': 'CSKS', 'sinkFolder': 'bronze/sap_finance/csks', 'filePrefix': 'csks', 'columnList': '[MANDT],[KOKRS],[KOSTL],[BUKRS],[PRCTR],[VERAK],[KOSAR],[KHINR],[DATAB],[DATBI]', 'sourceObject': '[SAPABAP1].[CSKS]'}, {'name': 'CEPC', 'sinkFolder': 'bronze/sap_finance/cepc', 'filePrefix': 'cepc', 'columnList': '[MANDT],[KOKRS],[PRCTR],[BUKRS],[VERAK],[KHINR],[SEGMENT],[DATAB],[DATBI]', 'sourceObject': '[SAPABAP1].[CEPC]'}, {'name': 'SKA1', 'sinkFolder': 'bronze/sap_finance/ska1', 'filePrefix': 'ska1', 'columnList': '[MANDT],[KTOPL],[SAKNR],[XBILK],[GVTYP],[KTOKS]', 'sourceObject': '[SAPABAP1].[SKA1]'}, {'name': 'SKAT', 'sinkFolder': 'bronze/sap_finance/skat', 'filePrefix': 'skat', 'columnList': '[MANDT],[SPRAS],[KTOPL],[SAKNR],[TXT20],[TXT50]', 'sourceObject': '[SAPABAP1].[SKAT]'}, {'name': 'LFA1', 'sinkFolder': 'bronze/sap_finance/lfa1', 'filePrefix': 'lfa1', 'columnList': '[MANDT],[LIFNR],[NAME1],[NAME2],[LAND1],[ORT01],[REGIO],[KTOKK],[STCD1]', 'sourceObject': '[SAPABAP1].[LFA1]'}, {'name': 'KNA1', 'sinkFolder': 'bronze/sap_finance/kna1', 'filePrefix': 'kna1', 'columnList': '[MANDT],[KUNNR],[NAME1],[NAME2],[LAND1],[ORT01],[REGIO],[KTOKD],[STCD1]', 'sourceObject': '[SAPABAP1].[KNA1]'}, {'name': 'FAGLFLEXP', 'sinkFolder': 'bronze/sap_finance/faglflexp', 'filePrefix': 'faglflexp', 'columnList': '[RCLNT],[RLDNR],[RBUKRS],[RYEAR],[RACCT],[PRCTR],[KOKRS],[VERSN],[CURTYPE],[RHCUR],[TSL01],[TSL02],[TSL03],[TSL04],[TSL05],[TSL06],[TSL07],[TSL08],[TSL09],[TSL10],[TSL11],[TSL12],[TSL13],[TSL14],[TSL15],[TSL16]', 'sourceObject': '[SAPABAP1].[FAGLFLEXP]'}, {'name': 'BUDGET', 'sinkFolder': 'bronze/sap_finance/budget', 'filePrefix': 'budget', 'columnList': '[COMPANY_CODE],[GL_ACCOUNT],[COST_CENTER],[PROFIT_CENTER],[FISCAL_YEAR],[FISCAL_PERIOD],[BUDGET_AMOUNT_LOCAL],[FORECAST_AMOUNT_LOCAL],[PLAN_VERSION],[DATA_CLASSIFICATION]', 'sourceObject': '[SAPDEMO].[BUDGET]'}, {'name': 'DIVISION_REFERENCE', 'sinkFolder': 'bronze/sap_finance/division_reference', 'filePrefix': 'division_reference', 'columnList': '[DIVISION_CODE],[DIVISION_NAME]', 'sourceObject': '[SAPDEMO].[DIVISION_REFERENCE]'}, {'name': 'GL_ACCOUNT_REFERENCE', 'sinkFolder': 'bronze/sap_finance/gl_account_reference', 'filePrefix': 'gl_account_reference', 'columnList': '[GL_ACCOUNT],[GL_ACCOUNT_NAME],[FINANCE_CATEGORY]', 'sourceObject': '[SAPDEMO].[GL_ACCOUNT_REFERENCE]'}]
BRONZE_ROOT = "Files/bronze/sap_finance"

for parameter_name, parameter_value in {
    "sql_server": sql_server,
    "sql_database": sql_database,
    "processing_date": processing_date,
}.items():
    if not parameter_value.strip():
        raise ValueError(f"{parameter_name} is required")

try:
    run_date = datetime.strptime(processing_date, "%Y-%m-%d").date()
except ValueError as exc:
    raise ValueError("processing_date must use yyyy-MM-dd format") from exc
date_key = run_date.strftime("%Y%m%d")

access_token = notebookutils.credentials.getToken("https://database.windows.net/")
jdbc_url = (
    f"jdbc:sqlserver://{sql_server};"
    f"databaseName={sql_database};encrypt=true;trustServerCertificate=false;"
)
run_id = uuid.uuid4().hex
results = []

for item in TABLE_CONFIG:
    table_name = item["name"].lower()
    query = f"SELECT {item['columnList']} FROM {item['sourceObject']}"
    frame = (spark.read.format("jdbc")
             .option("url", jdbc_url)
             .option("query", query)
             .option("accessToken", access_token)
             .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver")
             .load())
    row_count = frame.count()
    if row_count == 0:
        raise RuntimeError(f"{item['name']} returned zero rows")

    folder = f"{BRONZE_ROOT}/{table_name}"
    staging = f"{folder}/_staging_{run_id}"
    destination = f"{folder}/{item['filePrefix']}_{date_key}.csv"
    (frame.coalesce(1).write.mode("overwrite")
     .option("header", True).option("quoteAll", True).csv(staging))
    part_files = [
        file_info for file_info in notebookutils.fs.ls(staging)
        if PurePosixPath(file_info.path).name.startswith("part-")
        and file_info.path.endswith(".csv")
    ]
    if len(part_files) != 1:
        raise RuntimeError(f"Expected one CSV part for {item['name']}, found {len(part_files)}")
    if notebookutils.fs.exists(destination):
        notebookutils.fs.rm(destination, False)
    notebookutils.fs.mv(part_files[0].path, destination, True)
    notebookutils.fs.rm(staging, True)
    if not notebookutils.fs.exists(destination):
        raise RuntimeError(f"Bronze file was not created: {destination}")
    results.append((item["name"], row_count, destination))
    print(f"{item['name']}: rows={row_count}, file={destination}")

if len(results) != len(TABLE_CONFIG):
    raise RuntimeError("Not all source snapshots were written")
print(f"Created {len(results)} full Bronze snapshots for {processing_date}")
